In [ ]:
import jax
import jax.numpy as jnp

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, Predictive

import matplotlib.pyplot as plt

## Define and show model

In [ ]:
def model(data=None):
    p = numpyro.sample("p", dist.Beta(1., 1.))

    numpyro.sample("obs", dist.Bernoulli(p), obs=data)

In [ ]:
numpyro.render_model(model, render_distributions=True, )

## Sample from my model without data

In [ ]:
key = jax.random.PRNGKey(0)

predictive = Predictive(model, num_samples=10)
samples = predictive(key)

print(samples)

## Several iid coin throws

In [ ]:
def model(data=None, n=10):
    p = numpyro.sample("p", dist.Beta(1., 1.))

    with numpyro.plate("throws", n):
        numpyro.sample("obs", dist.Bernoulli(p), obs=data)

In [ ]:
numpyro.render_model(model, render_distributions=True, )

In [ ]:
key = jax.random.PRNGKey(0)

predictive = Predictive(model, num_samples=5)
samples = predictive(key, n=10)
print(samples)

## Doing inference from data

In [ ]:
data = jnp.array([0, 1, 0, 1, 1, 0, 1, 0, 1, 1])

In [ ]:
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=10000)
mcmc.run(jax.random.PRNGKey(0), data=data, n=len(data))
mcmc.print_summary()

In [ ]:
posterior_samples = mcmc.get_samples()


In [ ]:

plt.figure()
plt.hist(posterior_samples["p"], bins=40, density=True, alpha=0.7)
plt.xlabel("p")
plt.ylabel("density")
plt.show()

## Sample from the posterior predictive distribution

In [ ]:
posterior_predictive = Predictive(model, posterior_samples=posterior_samples)
new_samples = posterior_predictive(jax.random.PRNGKey(1), n=10)